# Quantum harmonic oscillator

*quantum · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/harmonic_oscillator)

The Hamiltonian $H = -\tfrac12\,d^2/dx^2 + \tfrac12\omega^2x^2$ becomes a matrix on a grid, and torch.linalg.eigh returns its evenly spaced energy ladder $E_n = \omega(n + \tfrac12)$. A displaced Gaussian (a coherent state) is written as a sum of eigenstates; giving each one its phase $e^{-iE_n t}$ makes the packet swing back and forth exactly like a classical mass on a spring.

**Vector form**

$$\hat H\,\psi = -\tfrac12\,\frac{d^2\psi}{dx^2} + \tfrac12\,\omega^2x^2\,\psi = E\,\psi,\qquad \psi(x,t) = \sum_n c_n\,\varphi_n(x)\,e^{-iE_n t}$$

**Index (tensor) form**

$$H_{xy} = -\tfrac12\,D2_{xy} + V_x\,\delta_{xy},\qquad H_{xy}\,\varphi_{yn} = E_n\,\varphi_{xn},\qquad \psi_{x}(t) = \varphi_{xn}\,c_n\,e^{-iE_n t}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_x$ | `potential(x, omega)` | (N,) | spring potential $\tfrac12\omega^2x^2$ at every grid point |
| $H_{xy}$ | `hamiltonian(x, dx, omega)` | (N, N) | kinetic $-\tfrac12\,D2$ plus the potential on the diagonal |
| $E_n,\ \varphi_{xn}$ | `torch.linalg.eigh(H)` | (N,), (N, N) | energies and eigenvectors (column $n$ is state $n$) |
| $c_n$ | `einsum('xn,x->n', phi, psi0) * dx` | (N,) | repeated $x$ ⇒ summed: overlap of the start state with state $n$ |
| $\psi_x(t)$ | `einsum('xn,n->x', phi, c * exp(-1j*E*t))` | (N,) | repeated $n$ ⇒ summed: rebuild $\psi$ from its phased components |
| $\langle x\rangle$ | `einsum('x,x,x->', psi.conj(), x, psi) * dx` | () | average position, compared with the classical $x_0\cos\omega t$ |

Units $\hbar = m = 1$. The grid is a large box with $\psi = 0$ at its walls, far enough out that the walls never matter. The faint curves are the lowest stationary states $\varphi_n(x)$, drawn at the height of their energy $E_n$; the spacing between rungs is exactly $\omega$. The start state is the ground-state Gaussian shifted to $x_0$. Its weights $\lvert c_n\rvert^2$ form a Poisson distribution with mean $\omega x_0^2/2$, and because all the energies differ by whole multiples of $\omega$, the packet returns to $x_0$ after every period $2\pi/\omega$ without spreading. $\langle x\rangle$ follows the classical motion $x_0\cos\omega t$ (Ehrenfest's theorem is exact for a quadratic potential).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/harmonic_oscillator/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import mean_position, norm


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    omega, x0 = params["omega"], params["x0"]
    L = 2 * (x0 + 7 / math.sqrt(omega))  # box much wider than the swing plus a few ground-state widths
    x, dx = tensors.grid(params["n_grid"], L, dev)
    H = tensors.hamiltonian(x, dx, omega)

    energies, phi = solve.eigenstates(H, dx)
    c = solve.coefficients(phi, tensors.coherent_state(x, x0, omega), dx)
    period = 2 * math.pi / omega
    times, psi = solve.solve(phi, energies, c, params["periods"] * period)

    for n, E in enumerate(energies[:4].tolist()):
        print(f"E_{n} = {E:8.4f}   exact omega*(n + 1/2) = {omega * (n + 0.5):8.4f}")
    x_mean = mean_position(psi, x, dx)
    t_end = times[-1].item()
    print(f"<x> at t_max = {x_mean[-1].item():.4f}   classical x0 cos(wt) = {x0 * math.cos(omega * t_end):.4f}")
    print(f"norm at t_max: {norm(psi[-1], dx):.8f}")
    return plot.animate(x, times, psi, phi, energies, x_mean, omega, x0, out_dir / "harmonic_oscillator.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/harmonic_oscillator/equation.py
r"""Physics → tensor translation.

Time-independent Schrödinger equation for a mass on a spring (units ħ = m = 1):

    \hat H \varphi = -\tfrac12 \varphi'' + \tfrac12 \omega^2 x^2 \varphi = E \varphi.

On N grid points the operator becomes a matrix and the wavefunction a vector:

    H_{xy} = -\tfrac12 D2_{xy} + V_x \delta_{xy},      V_x = \tfrac12 \omega^2 x_x^2,

and collecting all eigenvectors as the columns of \varphi_{xn} (grid point x, state n):

    H_{xy} \varphi_{yn} = E_n \varphi_{xn}             (y summed).

Time evolution: each eigenstate only rotates its phase, so

    c_n = \varphi_{xn} \psi_x(0) \, dx                  (x summed: projection)
    \psi_x(t) = \varphi_{xn} \, c_n e^{-i E_n t}        (n summed: superposition)
    \langle x \rangle = \psi^*_x \, x_x \, \psi_x \, dx    (x summed)
"""

import torch


def project(phi: torch.Tensor, psi0: torch.Tensor, dx: float) -> torch.Tensor:
    r"""c_n = \varphi_{xn} \psi_x dx.   phi: (N, M), psi0: (N,) complex → (M,) complex"""
    return torch.einsum("xn,x->n", phi.to(psi0.dtype), psi0) * dx


def evolve(phi: torch.Tensor, energies: torch.Tensor, c: torch.Tensor, t: float) -> torch.Tensor:
    r"""\psi_x(t) = \varphi_{xn} c_n e^{-i E_n t}.   phi: (N, M), energies: (M,), c: (M,) → (N,) complex"""
    phase = torch.exp(-1j * energies * t)  # (M,)  e^{-i E_n t}
    return torch.einsum("xn,n->x", phi.to(c.dtype), c * phase)


def mean_position(psi: torch.Tensor, x: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\langle x \rangle = \psi^*_x x_x \psi_x dx  (x summed).   psi: (..., N), x: (N,) → (...)"""
    return torch.einsum("...x,x,...x->...", psi.conj(), x.to(psi.dtype), psi).real * dx


def norm(psi: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\int |\psi|^2 dx \approx \psi^*_x \psi_x dx  (x summed).   (..., N) → (...)"""
    return torch.einsum("...x,...x->...", psi.conj(), psi).real * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/harmonic_oscillator/tensors.py
"""Build the concrete tensors: the grid, the spring potential, the Hamiltonian matrix and the start state."""

import math

import torch


def grid(n: int, L: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """N interior points of the box [−L/2, L/2]; the walls are left out (ψ = 0 there). Returns (x (N,), dx)."""
    dx = L / (n + 1)
    x = -L / 2 + dx * torch.arange(1, n + 1, dtype=torch.float64, device=device)  # (N,)
    return x, dx


def potential(x: torch.Tensor, omega: float) -> torch.Tensor:
    """V_x = ½ ω² x². Shape (N,)."""
    return 0.5 * omega**2 * x**2


def second_derivative(n: int, dx: float, device: torch.device) -> torch.Tensor:
    """D2_{xy}: −2 on the diagonal, 1 on the two neighbouring diagonals, all over dx². Shape (N, N)."""
    main = -2.0 * torch.ones(n, dtype=torch.float64, device=device)  # (N,)
    side = torch.ones(n - 1, dtype=torch.float64, device=device)  # (N-1,)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / dx**2


def hamiltonian(x: torch.Tensor, dx: float, omega: float) -> torch.Tensor:
    """H_{xy} = −½ D2_{xy} + V_x δ_{xy}. Shape (N, N), symmetric."""
    kinetic = -0.5 * second_derivative(len(x), dx, x.device)  # (N, N)
    return kinetic + torch.diag(potential(x, omega))  # V only on the diagonal


def coherent_state(x: torch.Tensor, x0: float, omega: float) -> torch.Tensor:
    """ψ_x(0) = (ω/π)^¼ exp(−ω(x − x0)²/2): the ground state shifted to x0, at rest. Shape (N,) complex."""
    psi = (omega / math.pi) ** 0.25 * torch.exp(-omega * (x - x0) ** 2 / 2)  # (N,) real
    return psi.to(torch.complex128)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/harmonic_oscillator/solve.py
"""Diagonalise H once, then evolve the coherent state exactly in the eigenbasis."""

import math

import torch

from .equation import evolve, project


def eigenstates(H: torch.Tensor, dx: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (E_n (N,), φ_{xn} (N, N)) sorted by energy, normalised so Σ_x φ_{xn}² dx = 1.

    eigh's vectors have Σ_x φ² = 1; dividing by √dx gives the continuum normalisation ∫|φ|² dx = 1.
    """
    energies, phi = torch.linalg.eigh(H)
    return energies, phi / math.sqrt(dx)


def coefficients(phi: torch.Tensor, psi0: torch.Tensor, dx: float) -> torch.Tensor:
    """c_n of the start state in the eigenbasis: (N,) complex."""
    return project(phi, psi0, dx)


def solve(
    phi: torch.Tensor, energies: torch.Tensor, c: torch.Tensor, t_max: float, n_steps: int = 240
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, N) complex). No time stepping: every t is evaluated exactly."""
    times = torch.linspace(0.0, t_max, n_steps + 1, dtype=torch.float64)
    psi = torch.stack([evolve(phi, energies, c, t) for t in times.tolist()])  # (T, N)
    return times, psi


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/harmonic_oscillator/plot.py
"""Animate |ψ|² swinging in the parabola, with the lowest eigenstates drawn faintly at their energies."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    x: torch.Tensor,
    times: torch.Tensor,
    psi: torch.Tensor,
    phi: torch.Tensor,
    energies: torch.Tensor,
    x_mean: torch.Tensor,
    omega: float,
    x0: float,
    path: Path,
    n_show: int = 6,
) -> Path:
    xs = x.cpu().numpy()
    prob = (psi.abs() ** 2).cpu().numpy()  # (T, N)
    t = times.cpu().numpy()
    xm = x_mean.cpu().numpy()  # (T,)
    E = energies[:n_show].cpu().numpy()
    states = phi[:, :n_show].T.cpu().numpy()  # (n_show, N)

    x_view = abs(x0) + 4 / math.sqrt(omega)
    e_top = max(E[-1] + omega, 0.5 * omega**2 * x0**2 * 1.2)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(-x_view, x_view)
    ax.set_ylim(0, e_top)
    ax.set_xlabel("x")
    ax.set_ylabel("energy  (V(x), E_n)")
    ax.plot(xs, 0.5 * omega**2 * xs**2, color="#6b7280", lw=2, label="V = ½ω²x²")
    scale = 0.35 * omega / abs(states).max()  # eigenstate wiggles fit between the rungs
    for En, state in zip(E, states):  # faint stationary states φ_n(x) drawn at height E_n
        ax.axhline(En, color="#e5e7eb", lw=0.8)
        ax.plot(xs, En + scale * state, color="#9ca3af", lw=0.8, ls="--")

    # |ψ|² on a second axis so its height is independent of the energy scale
    ax_p = ax.twinx()
    ax_p.grid(False)
    ax_p.set_ylim(0, 1.6 * prob.max())
    ax_p.set_ylabel("|ψ|²", color=ACCENT[0])
    (line,) = ax_p.plot([], [], color=ACCENT[0], lw=2, label="|ψ|²")
    fill = [ax_p.fill_between(xs, 0 * xs, color=ACCENT[0], alpha=0.2)]
    (mean_dot,) = ax_p.plot([], [], "v", color=ACCENT[0], ms=9, label="mean position <x>")
    (ball,) = ax_p.plot([], [], "o", color=ACCENT[1], ms=8, label="classical x0 cos ωt")
    clock = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top", family="monospace")
    ax_p.legend(loc="upper right", fontsize=8)

    def update(step: int):
        line.set_data(xs, prob[step])
        fill[0].remove()
        fill[0] = ax_p.fill_between(xs, prob[step], color=ACCENT[0], alpha=0.2)
        mean_dot.set_data([xm[step]], [1.35 * prob.max()])
        ball.set_data([x0 * math.cos(omega * t[step])], [1.45 * prob.max()])
        clock.set_text(f"t = {t[step]:5.2f}\n<x> = {xm[step]:+.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'omega': 1.0,
    'x0': 2.5,
    'periods': 2.0,
    'n_grid': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.harmonic_oscillator")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.harmonic_oscillator.main")
problem_dir = Path("problems/quantum/harmonic_oscillator")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
